# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

<div style="background-color:#f1be3e">

_Fill in your group number **from Brightspace**, names, and student numbers._
    
|    Group   |           X          |
|------------|----------------------|
| Student A  |        XXXXXXX       |
| Student B  |        XXXXXXX       |
| Student C  |        XXXXXXX       |
| Student D  |        XXXXXXX       |

#### Imports

In [ ]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

In [69]:
np.random.seed(42)

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 2

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 3

<div style="background-color:#f1be3e">

_Write your answer here._

### 1.2 Genetic Algorithm

In [78]:
class Chromosome:

    def __init__(self, size):
        self.size = size
        self.products = np.random.permutation(size)
        self._fitness_cache = None
    
    def fitness(self, tsp_data):
        if self._fitness_cache is not None:
            return self._fitness_cache
        
        result = tsp_data.start_to_product[self.products[0]].size() + tsp_data.product_to_end[self.products[-1]].size()

        for i in range(0, self.size - 1):
            j1 = self.products[i]
            j2 = self.products[i + 1]

            result += tsp_data.product_to_product[j1][j2].size()
        
        self._fitness_cache = result
        return result

    def copy(self):
        chromosome = Chromosome(self.size)
        chromosome.products = self.products.copy()
        chromosome._fitness_cache = self._fitness_cache
        return chromosome

    def mutation(self, p_mutation, p_swap_or_inversion):
        mutated = self.copy()

        if np.random.rand() <= p_mutation:
            if np.random.rand() <= p_swap_or_inversion:
                return mutated.modification_swap()
            else:
                return mutated.modification_inversion()

        return mutated
    
    def modification_swap(self):
        i1, i2 = np.random.choice(self.size, size=2, replace=False)

        aux = self.products[i1]
        self.products[i1] = self.products[i2]
        self.products[i2] = aux
        self._fitness_cache = None

        return self

    def modification_inversion(self):
        if self.size < 2:
            return self

        i1, i2 = np.random.choice(self.size, size=2, replace=False)

        if i2 < i1:
            aux = i1
            i1 = i2
            i2 = aux

        self.products[i1:i2 + 1] = self.products[i1:i2 + 1][::-1]
        self._fitness_cache = None

        return self

In [79]:
def pmx(parent1, parent2):
    size = parent1.size
    p1 = parent1.products
    p2 = parent2.products

    cut1, cut2 = np.sort(np.random.choice(size, 2, replace=False))

    def build_child(donor, receiver):
        child = np.full(size, -1, dtype=donor.dtype)

        # Copy PMX segment from donor parent
        child[cut1:cut2 + 1] = donor[cut1:cut2 + 1]
        donor_segment = donor[cut1:cut2 + 1]

        for i in range(size):
            if cut1 <= i <= cut2:
                continue

            candidate = receiver[i]
            steps = 0

            # Resolve conflicts by following PMX mapping across segment positions.
            while candidate in donor_segment and steps <= size:
                mapped_idx = np.where(donor == candidate)[0][0]
                candidate = receiver[mapped_idx]
                steps += 1

            child[i] = candidate

        # Safety fill for any rare unresolved slots.
        if np.any(child == -1):
            used = set(child[child != -1].tolist())
            missing = [gene for gene in donor if gene not in used]
            miss_idx = 0
            for i in range(size):
                if child[i] == -1:
                    child[i] = missing[miss_idx]
                    miss_idx += 1

        return child

    child1_products = build_child(p1, p2)
    child2_products = build_child(p2, p1)

    child1 = parent1.copy()
    child2 = parent2.copy()
    child1.products = child1_products
    child2.products = child2_products

    # Invalidate optional cached fitness values, if present
    if hasattr(child1, "_fitness_cache"):
        child1._fitness_cache = None
    if hasattr(child2, "_fitness_cache"):
        child2._fitness_cache = None

    return child1, child2

In [82]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    """
    def __init__(self, generations, pop_size, p_elitism = 0.05, p_crossover = 0.25, p_mutation = 0.2, p_swap_or_inversion = 0.6, early_stop = 50):
        self.generations = generations
        self.pop_size = pop_size

        self.p_elitism = p_elitism

        self.p_crossover = p_crossover
        self.p_mutation = p_mutation
        self.p_swap_or_inversion = p_swap_or_inversion

        self.early_stop = early_stop
    
    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        nr_of_products = len(tsp_data.product_locations)
        chromosomes = self.initial_generation(nr_of_products, tsp_data)

        best_fitness = chromosomes[0].fitness(tsp_data)
        no_improve_count = 0

        for _ in range(self.generations):
            chromosomes = self.assemble_next(chromosomes, tsp_data)
            current_best = chromosomes[0].fitness(tsp_data)

            if current_best < best_fitness:
                best_fitness = current_best
                no_improve_count = 0
            else:
                no_improve_count += 1

            if self.early_stop is not None and self.early_stop > 0 and no_improve_count >= self.early_stop:
                break
    
        return chromosomes[0].products

    def roulette(self, chromosomes, tsp_data):
        fitness_values = np.array([c.fitness(tsp_data) for c in chromosomes], dtype=float)

        scores = 1.0 / (fitness_values + 1e-12)
        probabilities = scores / np.sum(scores)
        cumulative = np.cumsum(probabilities)

        n1 = np.random.rand()
        n2 = np.random.rand()

        i1 = np.searchsorted(cumulative, n1, side="left")
        i2 = np.searchsorted(cumulative, n2, side="left")
        i1 = min(i1, len(chromosomes) - 1)
        i2 = min(i2, len(chromosomes) - 1)

        return chromosomes[i1].copy(), chromosomes[i2].copy()


    def crossover(self, c_1, c_2):
        crossed_1 = c_1 
        crossed_2 = c_2

        if np.random.rand() < self.p_crossover:
            crossed_1, crossed_2 = pmx(crossed_1, crossed_2)

        return crossed_1.mutation(self.p_mutation, self.p_swap_or_inversion), crossed_2.mutation(self.p_mutation, self.p_swap_or_inversion)

    
    def assemble_next(self, chromosomes, tsp_data):
        # Copy top performers
        elite_count = max(1, int(self.pop_size * self.p_elitism))
        new_chromosomes = [chromosomes[i].copy() for i in range(elite_count)]

        while len(new_chromosomes) < self.pop_size:

            parent_1, parent_2 = self.roulette(chromosomes, tsp_data)
            offspring_1, offspring_2 = self.crossover(parent_1, parent_2)
            new_chromosomes.append(offspring_1)
            if len(new_chromosomes) < self.pop_size:
                new_chromosomes.append(offspring_2)
        
        new_chromosomes.sort(key = lambda c: c.fitness(tsp_data))
        return new_chromosomes[:self.pop_size]
        

    # At the moment this only creates completely random chromosomes, maybe add some better options with closest neighbours
    def initial_generation(self, chromosome_length, tsp_data):
        chromosomes = []

        for _ in range(self.pop_size):
            chromosomes.append(Chromosome(chromosome_length))

        chromosomes.sort(key = lambda c: c.fitness(tsp_data))

        return chromosomes



#### Question 4

<div style="background-color:#ffffff">

Our genes represent permutations for paths, giving us a path to follow from one product to another, in a certain order. Therefore, we encode our phenotypes as an array **a** of non-repeating values, the number in i-th position representing that the i-th product we go to is **a[i]**.

#### Question 5

<div style="background-color:#ffffff">

Our fitness function is the total length of the path selected by the phenotype, meaning the sum of the distances from **start** to the first product, from product i to product i + 1 for every i between 0 and n - 1, and from the location of product n to the end. This is a suitable fitness function because it's very easy to compute with our encoding and also fully represents the goal of our optimisation.

#### Question 6

<div style="background-color:#ffffff">

Within our design, parents are selected using the Roulette Wheel Selection method. Each chromosome has a probability proportional to its fitness to be chosen, divided by the total sum of fitness to normalize it. We choose 2 numbers between 0 and 1 and pick 2 parents based on what numbers the values lie in.

#### Question 7

<div style="background-color:#ffffff">

Genetic-wise, we use Crossovers, Swaps and Inversions. 

For crossovers, we use Partial Mapped Crossover (PMX), that has been implemented in the above cells. It is a permutation-safe crossover, because we have to make sure that when we "mate" 2 parents we don't have repeating numbers. The way it works is a section is chosen in both parents, the offspring copy that section, and for the rest of the chromosome they try to get the respective values from the other parent, that they did NOT copy the section from. if the value that they try to add to their chromosome is repeating, they look at a section-to-section map to see what value can be added instead from the other parent. This can be visualized better in the algorithm or in the attached reference at the bottom of the file.

If the Mutation check passes, we decide whether to swap or to invert the offspring. For swapping, we select 2 indices and switch them around in the offspring. For inverting, we select a section and reverse it.

These genetic operations make sure that we have randomness in our solutions, therefore we always have the possibility of finding a better solution of we haven't found the optimal one yet.

#### Question 8

<div style="background-color:#ffffff">

We prevent the algorithm from being stuck in a local minima by having randomness in our algorithm. Like we mentioned in the above sections, using the Roulette Wheel Selection and the mentioned genetic operations we try many differnet iterations, therefore we can always find the global optimum.

#### Question 9

<div style="background-color:#ffffff">

Elitism is keeping the best **p%** of top performers for the next generation. This way, we can preserver the best solution found until now. We do use it in our algorithm, because that way we can keep building on it and we never lose it. If we hadn't used it, it would be impossible to converge to a close-to-optimal solution.

#### Question 10

In [83]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 50
generations = 500
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
if solution is not None:
    tsp_data.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#ffffff">

# COME BACK

Using the provided persisted TSP data in the code cell above, my genetic algorithm produced a route of length **1343**.

In order to check it's optimality, we visualize it using the Visualizer Helper. By observing it we see that the found solution is good enough.

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

<div style="background-color:#ffffff">

Ant Colony Optimisation mimics nature, it simulates an actual ant colony. In ACO, individual ant paths are simulated. Ants release pheromones in their trails, if we observe the pheromone patterns created by the colony we can observe ants are attracted to these pheromones. As a result, shorter paths contain more pheromones per unit length and thus are traveled through more. In the contrary, longer paths contain less pheromones per unit length, thus are traveled less. This allows us to find the shortest path in a maze.

Finally, ACO is used in optimisation problems such as vehicle routing, network routing and job scheduling.

#### Question 12

<div style="background-color:#ffffff">

· Dead Ends: ants waste steps here as dead ends lead nowhere, forcing the ants to go back on their steps to the diversion of the dead ended branch. Many dead ends in a maze can increase the time and difficulty of finding the finish line as they cause long delays and resource wasting.

· Loops: if ants get stuck in a loop they might circle it indefinitely, potentially becoming an ant death spiral.

· Large open areas: no clear paths, makes dispersed routes.

· Long corridors with no branches: ants must commit early.

#### Question 13

<div style="background-color:#ffffff">

Ants need to carry pheromones so other ants have a learning signal, if not they would do random movements and paths. These new paths that have residual pheromones released by previous ants may indicate there was something of interest there, so paths with more pheromones are taken more. Usually this can be seen in nature when long paths of ants form a line from their colony to a food source. 

Ants can carry pheromones from the colony. This means the pheromones they carry are deposited through the path they take before going back to the colony. Only ants that reach the end deposit pheromones. Lets define the total pheromones an ant can carry as Q. Lets further define the total length of the path taken by an ant as L. We assume the pheromones are dropped uniformly through the path. This means longer paths will have the ants pheromones dispersed while shorter paths will have it more concentrated.

The total pheromones at a cell X defined by τ is Δτ = Q/L.



#### Question 14

<div style="background-color:#ffffff">

Pheromone evaporation is needed to cleanse old paths that direct ants toward past points of interest or random paths that were followed at the start. Also, it allows the colony to forget bad solutions and escape local optima. And lastly, prevents pheromones from building up indefinitely.

The pheromone evaporation formula needs to contain a tunable evaporation rate defined as p. Furthermore it needs to contain the current pheromone level in the cell Δτ. A higher evaporation rate means pheromone levels will reduce more so we need an inverse relation between evaporation rate and pheromone levels.

Lastly, a pheromone floor level must be enforced so some paths are not permanently ignored. This is defined as τ-min. 

New pheromone levels of the cell are defined as τ new, in the pheromone evaporation formula 

τ-new = max(τ-old * (1 - p), τ-min).


### 2.3 Implementing the Ant Algorithm

In [74]:
DIRECTIONS = [Direction.north, Direction.east, Direction.south, Direction.west]

# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification, max_steps):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random
        self.max_steps = max_steps

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        # Initialize the route and current position
        route = Route(self.start)
        self.current_position = self.start

        # Loop until we reach the end coordinate or max steps exceeded
        while self.current_position != self.end:
            if route.size() >= self.max_steps:
                return None # Exceeded max steps, likely stuck in a loop

            # Calculate movement probabilities based on surrounding pheromone levels
            probabilities = self._calculate_probabilities(
                self.maze.get_surrounding_pheromone(self.current_position)
            )

            if probabilities is None:
                # Dead end — backtrack one step
                if route.size() == 0:
                    return None  # Stuck at start, truly no path
                last_dir = route.remove_last()
                # Move back: reverse the last direction
                opposite = {
                    Direction.north: Direction.south,
                    Direction.south: Direction.north,
                    Direction.east: Direction.west,
                    Direction.west: Direction.east,
                }

                # Move back to the previous position
                self.current_position = self.current_position.add_direction(opposite[last_dir])
                continue

            # Choose a direction based on the calculated probabilities
            direction = self.rand.choices(DIRECTIONS, weights=probabilities)[0]
            self.current_position = self.current_position.add_direction(direction)
            # Add the chosen direction to the route
            route.add(direction)

        return route

    """
    Calculate movement probabilities for each direction based on surrounding pheromone levels.
    Directions leading to walls or out-of-bounds positions have pheromone 0 and thus
    probability 0, making them never selected. If all neighbors are inaccessible,
    falls back to equal probability as a safety measure.
    @param surrounding_pheromone: SurroundingPheromone object at current position
    @return list of probabilities for [north, east, south, west]
    """
    def _calculate_probabilities(self, surrounding_pheromone):
        # Extract pheromone levels for each direction
        pheromones = [surrounding_pheromone.get(d) for d in DIRECTIONS]
        total = sum(pheromones)
        if total == 0:
            return None  # ant is completely stuck — no valid moves
        # Normalize to get probabilities
        return [p / total for p in pheromones]
            


In [75]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ants representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length, tau_0, tau_min):
        self.walls = np.array(walls)
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.tau_0 = tau_0
        self.tau_min = tau_min
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        # Initialize pheromones to tau_0 on accessible tiles and 0 on walls
        self.pheromones = np.zeros((self.width, self.length))
        for x in range(self.width):
            for y in range(self.length):
                if self.walls[x][y] == 1:
                    self.pheromones[x][y] = self.tau_0

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        # Deposit pheromone along the route. The amount deposited is q divided by the route length.
        deposit = q / route.size()
        pos = route.get_start()
        self.pheromones[pos.x][pos.y] += deposit
        
        # Iterate through the route and add pheromone to each position along the way
        for direction in route.get_route():
            pos = pos.add_direction(direction)
            self.pheromones[pos.x][pos.y] += deposit

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        # Evaporate pheromones by multiplying with (1 - rho) and applying a minimum threshold tau_min
        self.pheromones = np.maximum(self.pheromones * (1 - rho), self.tau_min)
        # walls stay 0 — apply floor only to accessible cells
        self.pheromones = np.where(self.walls, self.pheromones, 0)
        # assert self.pheromones[self.walls == 0].sum() == 0, "Wall has non-zero pheromone!"

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        north = self.get_pheromone(position.add_direction(Direction.north))
        south = self.get_pheromone(position.add_direction(Direction.south))
        east = self.get_pheromone(position.add_direction(Direction.east))
        west = self.get_pheromone(position.add_direction(Direction.west))
        return SurroundingPheromone(north, east, south, west)

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the position of interest
    @return the amount of pheromone at the specified position
    """
    def get_pheromone(self, pos):
        if self.in_bounds(pos) and self.walls[pos.x][pos.y] == 1:
            return self.pheromones[pos.x][pos.y]
        else:
            return 0

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path, tau_0, tau_min):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length, tau_0, tau_min)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [76]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    @param max_steps: maximum steps per ant before giving up
    @param no_improve_limit: number of generations without improvement before stopping
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation, max_steps, no_improve_limit):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation
        self.max_steps = max_steps
        self.no_improve_limit = no_improve_limit  
        self.convergence_history = [] 

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()
        best_route = None
        no_improve_count = 0
        self.convergence_history = []  

        # Loop over generations
        for gen in range(self.generations):
            routes = []
            improved = False

            # For each ant, find a route and update the best route if necessary
            for _ in range(self.ants_per_gen):
                ant = StandardAnt(self.maze, path_specification, self.max_steps)
                route = ant.find_route()
                if route is not None:
                    routes.append(route)
                    if best_route is None or route.shorter_than(best_route):
                        best_route = route
                        improved = True

            # Update pheromones based on the routes found by the ants
            self.maze.add_pheromone_routes(routes, self.q)
            # Evaporate pheromones after all ants have deposited
            self.maze.evaporate(self.evaporation)
            # Record convergence history
            self.convergence_history.append(best_route.size() if best_route else None)

            if improved:
                no_improve_count = 0
            else:
                no_improve_count += 1

            # Check for convergence: if no improvement for a certain number of generations, stop early
            if no_improve_count >= self.no_improve_limit:
                print(f"Converged at generation {gen}")
                break

        return best_route

### Hyper-parameter Tuning

In [77]:
ants_per_gen = 100
no_gen = 50
q = 1600
evap = 0.05
tau_0 = 1
tau_min = 0.1
max_steps = 10000
no_improve_limit = 50

'''
# Construct the optimization objects
maze = Maze.create_maze("./../data/toy_maze.txt", tau_0, tau_min)
spec = PathSpecification.read_coordinates("./../data/toy_coordinates.txt")

aco = AntColonyOptimization(maze, ants_per_gen, no_gen, q, evap, max_steps, no_improve_limit)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
if shortest_route is None:
    print("No route found within step limit.")
else:
    print("Route size: " + str(shortest_route.size()))
    shortest_route.write_to_file("./../data/toy_solution.txt")

# Construct the optimization objects
maze = Maze.create_maze("./../data/easy_maze.txt", tau_0, tau_min)
spec = PathSpecification.read_coordinates("./../data/easy_coordinates.txt")

aco = AntColonyOptimization(maze, ants_per_gen, no_gen, q, evap, max_steps, no_improve_limit)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
if shortest_route is None:
    print("No route found within step limit.")
else:
    print("Route size: " + str(shortest_route.size()))
    shortest_route.write_to_file("./../data/easy_solution.txt")

# Construct the optimization objects
maze = Maze.create_maze("./../data/medium_maze.txt", tau_0, tau_min)
spec = PathSpecification.read_coordinates("./../data/medium_coordinates.txt")

aco = AntColonyOptimization(maze, ants_per_gen, no_gen, q, evap, max_steps, no_improve_limit)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
if shortest_route is None:
    print("No route found within step limit.")
else:
    print("Route size: " + str(shortest_route.size()))
    shortest_route.write_to_file("./../data/medium_solution.txt")
'''

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt", tau_0, tau_min)
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")

aco = AntColonyOptimization(maze, ants_per_gen, no_gen, q, evap, max_steps, no_improve_limit)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
if shortest_route is None:
    print("No route found within step limit.")
else:
    print("Route size: " + str(shortest_route.size()))
    shortest_route.write_to_file("./../data/hard_solution.txt")


# Construct the optimization objects
maze = Maze.create_maze("./../data/insane_maze.txt", tau_0, tau_min)
spec = PathSpecification.read_coordinates("./../data/insane_coordinates.txt")

aco = AntColonyOptimization(maze, ants_per_gen, no_gen, q, evap, max_steps, no_improve_limit)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
if shortest_route is None:
    print("No route found within step limit.")
else:
    print("Route size: " + str(shortest_route.size()))
    shortest_route.write_to_file("./../data/insane_solution.txt")


Ready reading maze file ./../data/hard_maze.txt
Converged at generation 49
Time taken: 279.45
No route found within step limit.
Ready reading maze file ./../data/insane_maze.txt


KeyboardInterrupt: 

In [ ]:
# ─── Build the 3×2 test maze inline ───────────────────────────────────────
#   walls[x][y]:  x=col (0-2), y=row (0-1)
#   Row 0 (top):  0 1 0  → only (1,0) open
#   Row 1 (bot):  1 1 1  → all open   START=(0,1)  END=(2,1)
walls = [[0, 1],   # x=0: wall at y=0, open at y=1
         [1, 1],   # x=1: open at y=0, open at y=1
         [0, 1]]   # x=2: wall at y=0, open at y=1

TAU_0   = 1.0
TAU_MIN = 0.01
maze = Maze(walls, width=3, length=2, tau_0=TAU_0, tau_min=TAU_MIN)
start = Coordinate(0, 1)
end   = Coordinate(2, 1)
spec  = PathSpecification(start, end)

def check(name, got, expected):
    ok = abs(got - expected) < 1e-9 if isinstance(expected, float) else got == expected
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}: got={got!r}, expected={expected!r}")

# ─── TEST 1: pheromone initialisation ─────────────────────────────────────
# Accessible cells must start at tau_0; walls must be 0
check("init pheromone wall  (0,0)", maze.pheromones[0][0], 0.0)
check("init pheromone open  (1,0)", maze.pheromones[1][0], TAU_0)
check("init pheromone start (0,1)", maze.pheromones[0][1], TAU_0)
check("init pheromone end   (2,1)", maze.pheromones[2][1], TAU_0)

# ─── TEST 2: get_surrounding_pheromone at START (0,1) ─────────────────────
# North of (0,1) = (0,0) → wall  → 0
# South of (0,1) = (0,2) → OOB   → 0
# East  of (0,1) = (1,1) → open  → 1
# West  of (0,1) = (-1,1)→ OOB   → 0
sp = maze.get_surrounding_pheromone(start)
check("surr N at start", sp.get(Direction.north), 0.0)
check("surr S at start", sp.get(Direction.south), 0.0)
check("surr E at start", sp.get(Direction.east),  TAU_0)
check("surr W at start", sp.get(Direction.west),  0.0)

# ─── TEST 3: add_pheromone_route ──────────────────────────────────────────
# Route: East, East  (length=2, deposit = Q/2 per cell)
Q = 2.0
route = Route(start)
route.add(Direction.east)
route.add(Direction.east)
maze.add_pheromone_route(route, Q)

deposit = Q / route.size()   # = 1.0
check("pheromone after deposit (0,1)", maze.pheromones[0][1], TAU_0 + deposit)
check("pheromone after deposit (1,1)", maze.pheromones[1][1], TAU_0 + deposit)
check("pheromone after deposit (2,1)", maze.pheromones[2][1], TAU_0 + deposit)
check("wall untouched after deposit (0,0)", maze.pheromones[0][0], 0.0)

# ─── TEST 4: evaporate ───────────────────────────────────────────────────
# After deposit: open cells have pheromone = 2.0  (tau_0=1 + deposit=1)
# After evaporate(rho=0.5): new = max(2.0 * 0.5, 0.01) = 1.0
rho = 0.5
maze.evaporate(rho)
check("evaporate open cell (0,1)", maze.pheromones[0][1], 2.0 * (1 - rho))
check("evaporate open cell (1,1)", maze.pheromones[1][1], 2.0 * (1 - rho))
check("wall still 0 after evaporate", maze.pheromones[0][0], 0.0)

# ─── TEST 5: ant finds route on tiny maze ────────────────────────────────
maze.reset()   # back to tau_0 everywhere
ant = StandardAnt(maze, spec, max_steps=1000)
found = ant.find_route()
check("ant finds a route (not None)", found is not None, True)
if found is not None:
    check("route reaches end",
          start.add_direction(Direction.east).add_direction(Direction.east) == end
          if found.size() == 2 else True,   # optimal; longer is ok too
          True)
    print(f"       route size = {found.size()} (optimal is 2)")

# ─── TEST 6: full ACO on tiny maze ───────────────────────────────────────
maze.reset()
aco = AntColonyOptimization(maze, ants_per_gen=10, generations=20,
                             q=10, evaporation=0.1,
                             max_steps=500, no_improve_limit=5)
result = aco.find_shortest_route(spec)
check("ACO finds a route (not None)", result is not None, True)
if result is not None:
    print(f"       ACO best route size = {result.size()} (optimal is 2)")

[PASS] init pheromone wall  (0,0): got=np.float64(0.0), expected=0.0
[PASS] init pheromone open  (1,0): got=np.float64(1.0), expected=1.0
[PASS] init pheromone start (0,1): got=np.float64(1.0), expected=1.0
[PASS] init pheromone end   (2,1): got=np.float64(1.0), expected=1.0
[PASS] surr N at start: got=0, expected=0.0
[PASS] surr S at start: got=0, expected=0.0
[PASS] surr E at start: got=np.float64(1.0), expected=1.0
[PASS] surr W at start: got=0, expected=0.0
[PASS] pheromone after deposit (0,1): got=np.float64(2.0), expected=2.0
[PASS] pheromone after deposit (1,1): got=np.float64(2.0), expected=2.0
[PASS] pheromone after deposit (2,1): got=np.float64(2.0), expected=2.0
[PASS] wall untouched after deposit (0,0): got=np.float64(0.0), expected=0.0
[PASS] evaporate open cell (0,1): got=np.float64(1.0), expected=1.0
[PASS] evaporate open cell (1,1): got=np.float64(1.0), expected=1.0
[PASS] wall still 0 after evaporate: got=np.float64(0.0), expected=0.0
[PASS] ant finds a route (not None

### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [ ]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [ ]:
# Please keep your parameters for the synthesis part easily changeable here
ants_per_gen = 50
no_gen = 100
q = 1000
evap = 0.1
max_steps = 10000
no_improve_limit = 20
tau_0 = 1
tau_min = 0.01

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt", tau_0, tau_min)
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, ants_per_gen, no_gen, q, evap, max_steps, no_improve_limit)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

Ready reading maze file ./../data/hard_maze.txt


ZeroDivisionError: division by zero

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#ffffff">

**If you made use of any non-course resources, cite them below.**

https://medium.com/@becmjo/genetic-algorithms-and-the-travelling-salesman-problem-d10d1daf96a1
https://ultraevolution.org/blog/pmx_crossover/